# Choosing between auto-merge and manual review in Dependabot

> last_verified: 2026-09-25 · dependabot (no version researched this cycle; no externally-verifiable claims made)

## Purpose

This notebook compares the two review postures teams use for Dependabot update pull requests — auto-merge and manual review — so a team can decide which updates may merge on a green pipeline and which need a human reviewer. It covers how each posture handles change scope, test signal, and rollback risk, with a decision aid at the end.

## When to use

Use auto-merge for low-risk, well-covered updates: narrow-scope dependency bumps in repositories where the continuous-integration pipeline (lint, unit tests, build) runs on every update pull request and failures block the merge. Use manual review for broad-scope or high-blast-radius updates: major-version bumps, updates to shared libraries consumed by many services, lockfile restructures, and any repository where pipeline coverage is thin. Most setups mix the two: auto-merge for routine patch-level bumps, manual review for everything else.

## Prerequisites

- A repository with Dependabot update pull requests enabled, so dependency bumps arrive as ordinary pull requests.
- A required-status pipeline on those pull requests (at minimum a build plus the unit test suite), so an automatic merge still has a quality gate in front of it.
- A written review policy stating which update scopes each posture covers; the kit companions under `dependabot/` give starting points: `docs/dependabot-security-update-auto-merge.md`, `configs/monorepo-ecosystem-schedules-reviewers.yaml`, and `scripts/dependabot-alert-aggregation.py`.


In [ ]:
# Comparison matrix: operational properties of each review posture.
# Values are qualitative (lower/higher burden), not measured benchmarks.

POSTURES = {
    "auto-merge": {
        "human_step": "none after authoring the policy; the merge happens once required checks pass",
        "quality_gate": "the required-status pipeline (build plus tests) must be green",
        "throughput": "high: routine bumps clear without reviewer scheduling",
        "risk_surface": "a green-but-insufficient suite can let a breaking bump through",
        "audit_trail": "merge record plus pipeline run; reviewer identity is the policy, not a person",
    },
    "manual-review": {
        "human_step": "a reviewer inspects scope, changelog, and pipeline signal before approving",
        "quality_gate": "pipeline signal plus reviewer judgment on scope and blast radius",
        "throughput": "lower: every bump waits on reviewer availability",
        "risk_surface": "reviewer fatigue on routine bumps; inconsistent standards across reviewers",
        "audit_trail": "named approver on each merge; rationale lives in review comments",
    },
}

for name, props in POSTURES.items():
    print(f"== {name} ==")
    for key, value in props.items():
        print(f"  {key}: {value}")

assert set(POSTURES) == {"auto-merge", "manual-review"}
assert all("quality_gate" in p for p in POSTURES.values())
print("matrix OK")


## Steps

### 1. Understand what auto-merge asks of the repository

Auto-merge starts with the pipeline, not the merge button. Every Dependabot pull request must trigger the same build and test suite that a human-authored change would, and those checks must be required — a merge that proceeds on a skipped or absent suite is unreviewed by anyone, human or machine. The ongoing cost is suite maintenance: as coverage rots, the set of updates that are safe to auto-merge shrinks, so the policy scope should be re-checked whenever the suite changes.

### 2. Understand what manual review asks of the team

Manual review adds a scope judgment to the pipeline signal. The reviewer checks what the bump touches (one leaf dependency versus a shared framework), whether the upstream changelog flags breaking changes, and whether the pipeline actually exercised the affected paths. The trade-off is latency and attention — a queue of routine bumps competes with feature work, and rubber-stamping creeps in when most bumps are trivially safe.

### 3. Compare the failure modes side by side

- Auto-merge fails when the suite is green but insufficient: the bump passes every check and still breaks behavior the suite does not cover. Mitigation is narrowing auto-merge scope to well-covered, narrow bumps.
- Manual review fails when reviewers stop reading: routine approvals become reflex, and the posture costs reviewer time without buying judgment. Mitigation is sending only bumps that benefit from judgment to humans.
- Either way, keep a revert path that is as fast as the merge path — a small, well-understood revert procedure bounds the cost of a wrong call under either posture.

### 4. Apply the decision aid below

Encode the bump's properties (scope narrow or broad, suite coverage strong or thin, blast radius contained or shared) and let the aid recommend a posture. Mixed policies are normal and legitimate: the aid flags when each side wins rather than forcing a single global answer.


In [ ]:
def recommend(scope_narrow, suite_strong, blast_radius_contained):
    """Recommend a Dependabot review posture from three yes/no properties."""
    if not suite_strong:
        return "manual-review"  # no pipeline signal to back an automatic merge
    if scope_narrow and blast_radius_contained:
        return "auto-merge"
    return "manual-review"  # broad scope or shared blast radius wants judgment


CASES = [
    # (narrow, strong_suite, contained, expected)
    (True, True, True, "auto-merge"),  # routine patch-level bump, green suite
    (True, False, True, "manual-review"),  # thin suite: nothing backs auto-merge
    (False, True, True, "manual-review"),  # broad bump despite a good suite
    (True, True, False, "manual-review"),  # shared library: blast radius decides
    (False, False, False, "manual-review"),  # risky on every axis
]

for narrow, strong, contained, expected in CASES:
    got = recommend(narrow, strong, contained)
    status = "OK" if got == expected else "MISMATCH"
    print(f"{status}: narrow={narrow} strong_suite={strong} contained={contained} -> {got}")
    assert got == expected, (narrow, strong, contained, got)

print("decision aid OK")


## Verify

- Re-run both code cells: the matrix prints two postures with matching `quality_gate` entries, and the decision aid reports `OK` for all five cases.
- Cross-check the auto-merge posture against the kit companion `docs/dependabot-security-update-auto-merge.md` and the scheduling/reviewer shape in `configs/monorepo-ecosystem-schedules-reviewers.yaml`.
- Confirm the triage input for either posture with `scripts/dependabot-alert-aggregation.py`, which rolls up alerts by severity and package.

## Common errors

- Treating the postures as mutually exclusive. A repository that auto-merges routine bumps can still require manual review for major-version or shared-library updates; document which scope each posture covers.
- Auto-merging without required checks. If the pipeline can be skipped or is absent on update pull requests, auto-merge is unreviewed merging — make the checks required first.
- Sending every bump to manual review. Reviewer fatigue turns approvals into reflex; reserve human review for bumps where judgment changes the outcome.
- Letting the policy scope drift from suite reality. When coverage shrinks, previously safe auto-merge scopes need narrowing; re-check scope whenever the suite changes.

## References

- `dependabot/docs/dependabot-security-update-auto-merge.md` — auto-merge workflow companion.
- `dependabot/configs/monorepo-ecosystem-schedules-reviewers.yaml` — schedule and reviewer wiring companion.
- `dependabot/scripts/dependabot-alert-aggregation.py` — alert aggregation for triage input.
